In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers accelerate opencv-python pillow torch torchvision

In [5]:
import os
import cv2
import torch
from PIL import Image
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

VIDEO_PATH = "/content/drive/MyDrive/Projet ReDev/right.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/Projet ReDev/segmentation_output/right"
STRIDE = 10   # process 1 frame every 10 frames (much faster)

os.makedirs(OUTPUT_DIR + "/snapshots", exist_ok=True)
os.makedirs(OUTPUT_DIR + "/masks", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model...")

processor = AutoImageProcessor.from_pretrained(
    "facebook/mask2former-swin-large-mapillary-vistas-semantic"
)

model = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-large-mapillary-vistas-semantic"
).to(device)

model.eval()

print("Model loaded on", device)

cap = cv2.VideoCapture(VIDEO_PATH)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("Total frames:", frame_count)
print("FPS:", fps)

idx = 0
processed = 0

while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    if idx % STRIDE != 0:
        idx += 1
        continue

    print("Processing frame", idx)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)

    inputs = processor(images=pil_img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = processor.post_process_semantic_segmentation(
        outputs,
        target_sizes=[pil_img.size[::-1]]
    )[0]

    snap_path = f"{OUTPUT_DIR}/snapshots/frame_{idx:06d}.jpg"
    mask_path = f"{OUTPUT_DIR}/masks/mask_{idx:06d}.png"

    pil_img.save(snap_path)

    mask = pred.cpu().numpy().astype("uint8")
    Image.fromarray(mask).save(mask_path)

    processed += 1
    idx += 1

print("Finished.")

Loading model...


Loading weights:   0%|          | 0/782 [00:00<?, ?it/s]

Model loaded on cuda
Total frames: 12175
FPS: 29.97002997002997
Processing frame 0
Processing frame 10
Processing frame 20
Processing frame 30
Processing frame 40
Processing frame 50
Processing frame 60
Processing frame 70
Processing frame 80
Processing frame 90
Processing frame 100
Processing frame 110
Processing frame 120
Processing frame 130
Processing frame 140
Processing frame 150
Processing frame 160
Processing frame 170
Processing frame 180
Processing frame 190
Processing frame 200
Processing frame 210
Processing frame 220
Processing frame 230
Processing frame 240
Processing frame 250
Processing frame 260
Processing frame 270
Processing frame 280
Processing frame 290
Processing frame 300
Processing frame 310
Processing frame 320
Processing frame 330
Processing frame 340
Processing frame 350
Processing frame 360
Processing frame 370
Processing frame 380
Processing frame 390
Processing frame 400
Processing frame 410
Processing frame 420
Processing frame 430
Processing frame 440
Pr

In [2]:
import numpy as np
import cv2
from pathlib import Path

id2label = {
    0: "Bird",
    1: "Ground Animal",
    2: "Curb",
    3: "Fence",
    4: "Guard Rail",
    5: "Barrier",
    6: "Wall",
    7: "Bike Lane",
    8: "Crosswalk - Plain",
    9: "Curb Cut",
    10: "Parking",
    11: "Pedestrian Area",
    12: "Rail Track",
    13: "Road",
    14: "Service Lane",
    15: "Sidewalk",
    16: "Bridge",
    17: "Building",
    18: "Tunnel",
    19: "Person",
    20: "Bicyclist",
    21: "Motorcyclist",
    22: "Other Rider",
    23: "Lane Marking - Crosswalk",
    24: "Lane Marking - General",
    25: "Mountain",
    26: "Sand",
    27: "Sky",
    28: "Snow",
    29: "Terrain",
    30: "Vegetation",
    31: "Water",
    32: "Banner",
    33: "Bench",
    34: "Bike Rack",
    35: "Billboard",
    36: "Catch Basin",
    37: "CCTV Camera",
    38: "Fire Hydrant",
    39: "Junction Box",
    40: "Mailbox",
    41: "Manhole",
    42: "Phone Booth",
    43: "Pothole",
    44: "Street Light",
    45: "Pole",
    46: "Traffic Sign Frame",
    47: "Utility Pole",
    48: "Traffic Light",
    49: "Traffic Sign (Back)",
    50: "Traffic Sign (Front)",
    51: "Trash Can",
    52: "Bicycle",
    53: "Boat",
    54: "Bus",
    55: "Car",
    56: "Caravan",
    57: "Motorcycle",
    58: "On Rails",
    59: "Other Vehicle",
    60: "Trailer",
    61: "Truck",
    62: "Wheeled Slow",
    63: "Car Mount",
    64: "Ego Vehicle"
  }
def mask_to_percentages(mask):
    unique, counts = np.unique(mask, return_counts=True)
    total = mask.size

    data = {}
    for cls, count in zip(unique, counts):
        label = id2label.get(int(cls), None)
        if label:
            data[label] = (count / total) * 100

    return data

def calculate_IVW_levels(data):

    tree = data.get('Vegetation', 0)
    building = data.get('Building', 0)
    pavement = data.get('Sidewalk', 0) + data.get('Pedestrian Area', 0)
    road = data.get('Road', 0)
    car = data.get('Car', 0) + data.get('Truck', 0) + data.get('Bus', 0)
    fence = data.get('Fence', 0)
    obstacles = (
        data.get('Pole', 0)
        + data.get('Traffic Sign (Back)', 0)
        + data.get('Traffic Sign (Front)', 0)
        + data.get('Parking', 0)
        + data.get('Billboard', 0)
    )

    # G
    G_i = tree / 100
    if G_i < 0.07: G = 1
    elif G_i < 0.127: G = 2
    elif G_i < 0.175: G = 3
    elif G_i < 0.228: G = 4
    else: G = 5

    # C
    C_i = (obstacles + car) / 100
    if C_i >= 0.09: C = 1
    elif C_i >= 0.044: C = 2
    elif C_i >= 0.02: C = 3
    elif C_i >= 0.007: C = 4
    else: C = 5

    # S
    if (pavement + road + fence) == 0:
        S = 1
    else:
        S_i = (building + tree) / (pavement + road + fence)
        if S_i < 0.8 or S_i >= 2.864: S = 1
        elif S_i < 1.063 or S_i >= 2.21: S = 2
        elif S_i < 1.213 or S_i >= 1.884: S = 3
        elif S_i < 1.35 or S_i >= 1.656: S = 4
        else: S = 5

    # D
    if road == 0:
        D = 1
    else:
        D_i = (pavement + fence) / road
        if D_i < 0.48 or D_i >= 4: D = 1
        elif D_i < 0.65 or D_i >= 3.007: D = 2
        elif D_i < 0.806 or D_i >= 2.137: D = 3
        elif D_i < 0.953 or D_i >= 1.707: D = 4
        else: D = 5

    return G, C, S, D


def compute_IVW_score(data):
    G, C, S, D = calculate_IVW_levels(data)
    return 5 * (G + C + S + D)

In [3]:
def pv_ivw_method1(mask_folder):

    folder = Path(mask_folder)
    ivw_scores = []

    for file in sorted(folder.glob("*.png")):
        mask = cv2.imread(str(file), cv2.IMREAD_UNCHANGED)

        data = mask_to_percentages(mask)
        ivw = compute_IVW_score(data)

        ivw_scores.append(ivw)

    return np.mean(ivw_scores)
def pv_ivw_method2(mask_folder):

    folder = Path(mask_folder)

    accumulated = {}
    n_frames = 0

    for file in sorted(folder.glob("*.png")):
        mask = cv2.imread(str(file), cv2.IMREAD_UNCHANGED)

        data = mask_to_percentages(mask)

        for k, v in data.items():
            accumulated[k] = accumulated.get(k, 0) + v

        n_frames += 1

    # Average percentages
    avg_data = {k: v / n_frames for k, v in accumulated.items()}

    return compute_IVW_score(avg_data)

In [4]:
import yaml
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

def time_to_frame(t, fps):
    return int(time_to_seconds(t) * fps)
def extract_frame_number(filename):
    # mask_000120.png → 120
    return int(filename.stem.split("_")[-1])

In [5]:
def compute_segment_pv_ivw(mask_folder, start_frame, end_frame):

    folder = Path(mask_folder)

    ivw_list = []
    accumulated = {}
    n_frames = 0

    for file in sorted(folder.glob("*.png")):

        frame_id = extract_frame_number(file)

        if frame_id < start_frame or frame_id > end_frame:
            continue

        mask = cv2.imread(str(file), cv2.IMREAD_UNCHANGED)
        data = mask_to_percentages(mask)

        # --- Method 1
        ivw = compute_IVW_score(data)
        ivw_list.append(ivw)

        # --- Method 2 accumulation
        for k, v in data.items():
            accumulated[k] = accumulated.get(k, 0) + v

        n_frames += 1

    if n_frames == 0:
        return None, None

    # --- Method 1
    m1 = np.mean(ivw_list)

    # --- Method 2
    avg_data = {k: v / n_frames for k, v in accumulated.items()}
    m2 = compute_IVW_score(avg_data)

    return m1, m2

In [6]:
def process_segments(yaml_path, mask_folder, fps):

    with open(yaml_path, "r") as f:
        config = yaml.safe_load(f)

    results = []

    for seg in config["segments"]:

        start_frame = time_to_frame(seg["start_time"], fps)
        end_frame   = time_to_frame(seg["end_time"], fps)

        m1, m2 = compute_segment_pv_ivw(
            mask_folder,
            start_frame,
            end_frame
        )

        results.append({
            "segment_id": seg["segment_id"],
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "PV_IVW_M1": m1,
            "PV_IVW_M2": m2
        })

    return pd.DataFrame(results)

In [8]:
mask_folder = "/content/drive/MyDrive/Projet ReDev/segmentation_output/right/masks"
yaml_path   = "/content/drive/MyDrive/Projet ReDev/Yamls sous-ségments/360_segments.yaml"

fps = 6

df = process_segments(yaml_path, mask_folder, fps)

df.to_csv("pv_ivw_segments.csv", index=False)

print(df)

   segment_id start_time  end_time  PV_IVW_M1  PV_IVW_M2
0           1   00:00:00  00:01:42  38.225806         35
1           2   00:01:42  00:02:43  46.666667         40
2           3   00:02:43  00:03:25  42.500000         40
3           4   00:03:25  00:03:45  39.615385         40
4           5   00:03:45  00:04:35  28.548387         25
5           6   00:04:35  00:05:25  29.677419         30
6           7   00:05:25  00:06:25  30.405405         30


In [16]:
mask_folder = "/content/drive/MyDrive/Projet ReDev/segmentation_output/right/masks"
yaml_path   = "/content/drive/MyDrive/Projet ReDev/Yamls sous-ségments/360_segments.yaml"

fps = 6

df = process_segments(yaml_path, mask_folder, fps)

df.to_csv("pv_ivw_right.csv", index=False)


In [17]:
df

,segment_id,start_time,end_time,PV_IVW_M1,PV_IVW_M2
0,1,00:00:00,00:01:42,38.225806,35
1,2,00:01:42,00:02:43,46.666667,40
2,3,00:02:43,00:03:25,42.500000,40
3,4,00:03:25,00:03:45,39.615385,40
4,5,00:03:45,00:04:35,28.548387,25
5,6,00:04:35,00:05:25,29.677419,30
6,7,00:05:25,00:06:25,30.405405,30


In [2]:
import yaml
import pandas as pd
from scipy.stats import pearsonr
from pathlib import Path

# Chemins

yaml_dir = Path("/content/drive/MyDrive/Yamls sous-ségments") # tous les YAML participants
ivw_file = Path("/content/pv_ivw_front.csv") # PV-IVW segmenté

#Charger PV-IVW de la vidéo
df_ivw = pd.read_csv(ivw_file)  # colonnes: segment_id, start_time, end_time, PV_IVW_M1, PV_IVW_M2

# Charger et moyenniser walkability des participants

walk_data = []

for yaml_file in yaml_dir.glob("*.yaml"):
    with open(yaml_file, "r") as f:
        data = yaml.safe_load(f)

    for seg in data['segments']:
        walk_data.append({
            'segment_id': seg['segment_id'],
            'walkability': seg['walkability_rating']
        })

df_walk = pd.DataFrame(walk_data)

# Moyenne par segment
df_walk_avg = df_walk.groupby('segment_id', as_index=False).mean()
df_walk_avg.rename(columns={'walkability':'walkability_mean'}, inplace=True)

#Fusionner PV-IVW et walkability moyenne

df_merged = pd.merge(df_ivw, df_walk_avg, on='segment_id')

#Calculer corrélation et p-value

for method in ['PV_IVW_M1', 'PV_IVW_M2']:
    corr, pval = pearsonr(df_merged[method], df_merged['walkability_mean'])
    print(f"Method: {method} | Pearson corr: {corr:.3f} | p-value: {pval:.3f}")

#afficher le tableau fusionné

print("\nMerged table:")
print(df_merged[['segment_id','walkability_mean','PV_IVW_M1','PV_IVW_M2']])

Method: PV_IVW_M1 | Pearson corr: -0.712 | p-value: 0.073
Method: PV_IVW_M2 | Pearson corr: -0.237 | p-value: 0.608

Merged table:
   segment_id  walkability_mean  PV_IVW_M1  PV_IVW_M2
0           1          2.000000  51.612903         50
1           2          3.666667  43.750000         40
2           3          2.844444  39.038462         40
3           4          3.688889  33.461538         50
4           5          3.600000  29.838710         30
5           6          3.233333  31.129032         30
6           7          3.577778  35.810811         50


In [3]:
import yaml
import pandas as pd
from scipy.stats import pearsonr
from pathlib import Path

# -----------------------------
# Chemins
# -----------------------------
yaml_dir = Path("/content/drive/MyDrive/Yamls sous-ségments")

ivw_front = pd.read_csv("/content/pv_ivw_front.csv")
ivw_left  = pd.read_csv("/content/pv_ivw_left.csv")
ivw_right = pd.read_csv("/content/pv_ivw_right.csv")

# -----------------------------
# Fusion des 3 vues
# -----------------------------
df_ivw = ivw_front[['segment_id', 'PV_IVW_M1', 'PV_IVW_M2']].copy()

df_ivw = df_ivw.merge(
    ivw_left[['segment_id', 'PV_IVW_M1', 'PV_IVW_M2']],
    on='segment_id',
    suffixes=('_front', '_left')
)

df_ivw = df_ivw.merge(
    ivw_right[['segment_id', 'PV_IVW_M1', 'PV_IVW_M2']],
    on='segment_id'
)

# Renommer pour clarté
df_ivw.rename(columns={
    'PV_IVW_M1': 'PV_IVW_M1_right',
    'PV_IVW_M2': 'PV_IVW_M2_right'
}, inplace=True)

# -----------------------------
# Moyenne des 3 vues
# -----------------------------
df_ivw['PV_IVW_M1_mean3'] = (
    df_ivw['PV_IVW_M1_front'] +
    df_ivw['PV_IVW_M1_left'] +
    df_ivw['PV_IVW_M1_right']
) / 3

df_ivw['PV_IVW_M2_mean3'] = (
    df_ivw['PV_IVW_M2_front'] +
    df_ivw['PV_IVW_M2_left'] +
    df_ivw['PV_IVW_M2_right']
) / 3

# -----------------------------
# Charger walkability participants
# -----------------------------
walk_data = []

for yaml_file in yaml_dir.glob("*.yaml"):
    with open(yaml_file, "r") as f:
        data = yaml.safe_load(f)

    for seg in data['segments']:
        walk_data.append({
            'segment_id': seg['segment_id'],
            'walkability': seg['walkability_rating']
        })

df_walk = pd.DataFrame(walk_data)

# Moyenne par segment
df_walk_avg = df_walk.groupby('segment_id', as_index=False).mean()
df_walk_avg.rename(columns={'walkability': 'walkability_mean'}, inplace=True)

# -----------------------------
# Fusion finale
# -----------------------------
df_merged = pd.merge(df_ivw, df_walk_avg, on='segment_id')

# -----------------------------
# Corrélation
# -----------------------------
for method in ['PV_IVW_M1_mean3', 'PV_IVW_M2_mean3']:
    corr, pval = pearsonr(df_merged[method], df_merged['walkability_mean'])
    print(f"{method} → corr: {corr:.3f} | p-value: {pval:.3f}")

# -----------------------------
# Résultat
# -----------------------------
print("\nMerged table:")
print(df_merged[['segment_id', 'walkability_mean', 'PV_IVW_M1_mean3', 'PV_IVW_M2_mean3']])

PV_IVW_M1_mean3 → corr: -0.319 | p-value: 0.485
PV_IVW_M2_mean3 → corr: 0.102 | p-value: 0.828

Merged table:
   segment_id  walkability_mean  PV_IVW_M1_mean3  PV_IVW_M2_mean3
0           1          2.000000        45.456989        41.666667
1           2          3.666667        49.953704        46.666667
2           3          2.844444        41.602564        41.666667
3           4          3.688889        38.205128        43.333333
4           5          3.600000        37.741935        38.333333
5           6          3.233333        37.096774        38.333333
6           7          3.577778        35.450450        40.000000
